### LAB ASSIGNMENT

Name: Joseph Atem Deng Aruei

Nim: 244107020242

Step 1: Imports and file paths

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

data_dir = Path.cwd()

voice_path = data_dir / "voice (1).csv"
training_dir = data_dir / "training"
test_dir = data_dir / "test"

print("Working directory:", data_dir)
print("Voice CSV found:", voice_path.exists())
print("Training folder found:", training_dir.is_dir())
print("Test folder found:", test_dir.is_dir())

Working directory: d:\Machine_learning_course\JS06
Voice CSV found: True
Training folder found: True
Test folder found: True


Step 2: Load the voice CSV

In [4]:
voice_df = pd.read_csv(voice_path)

print("Dataset shape:", voice_df.shape)
print(voice_df["label"].value_counts())
voice_df.head()

Dataset shape: (3168, 21)
label
male      1584
female    1584
Name: count, dtype: int64


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


Step 3: Train and compare the six voice SVM models

In [5]:
X_voice = voice_df.drop(columns="label")
y_voice = voice_df["label"]

voice_results = []

for split_name, test_fraction in [("70:30", 0.30), ("80:20", 0.20)]:
    X_train, X_test, y_train, y_test = train_test_split(
        X_voice,
        y_voice,
        test_size=test_fraction,
        random_state=42,
        stratify=y_voice
    )

    for kernel in ["linear", "poly", "rbf"]:
        model = make_pipeline(
            StandardScaler(),
            SVC(kernel=kernel)
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        voice_results.append({
            "Split": split_name,
            "Kernel": kernel,
            "Accuracy": accuracy_score(y_test, predictions)
        })

voice_results_df = pd.DataFrame(voice_results)
voice_results_df["Accuracy"] = voice_results_df["Accuracy"].round(4)
voice_results_df

,Split,Kernel,Accuracy
0,70:30,linear,0.9790
1,70:30,poly,0.9600
2,70:30,rbf,0.9832
3,80:20,linear,0.9748
4,80:20,poly,0.9558
5,80:20,rbf,0.9826


Step 4: Load the extracted images and calculate HOG features

In [6]:
image_features = []
image_labels = []

image_extensions = {".jpg", ".jpeg", ".png"}

for split_dir in [training_dir, test_dir]:
    for label in ["day", "night"]:
        class_dir = split_dir / label

        image_paths = sorted(
            path for path in class_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in image_extensions
        )

        for image_path in image_paths:
            with Image.open(image_path) as image:
                image = image.convert("L").resize((64, 64))
                image_array = np.asarray(image)

            features = hog(
                image_array,
                orientations=9,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2),
                block_norm="L2-Hys"
            )

            image_features.append(features)
            image_labels.append(class_dir.name)

X_images = np.asarray(image_features)
y_images = np.asarray(image_labels)

print("Images loaded:", len(y_images))
print("Feature array shape:", X_images.shape)
print(pd.Series(y_images).value_counts())

Images loaded: 400
Feature array shape: (400, 1764)
day      200
night    200
Name: count, dtype: int64


Step 5: Split 80:20, tune the RBF model, and report accuracy

In [9]:
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_images,
    y_images,
    test_size=0.20,
    random_state=42,
    stratify=y_images
)

rbf_pipeline = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf")
)

parameter_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": ["scale", 0.001, 0.01]
}

grid_search = GridSearchCV(
    estimator=rbf_pipeline,
    param_grid=parameter_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_img, y_train_img)

image_predictions = grid_search.predict(X_test_img)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", round(grid_search.best_score_, 4))
print(
    "Held-out test accuracy:",
    round(accuracy_score(y_test_img, image_predictions), 4)
)

Best parameters: {'svc__C': 10, 'svc__gamma': 'scale'}
Best cross-validation accuracy: 0.9937
Held-out test accuracy: 1.0
